In [1]:
import numpy as np
import torch
import torchmetrics
from sklearn.datasets import load_sample_images
from torch.nn import CrossEntropyLoss

sample_images = np.stack(load_sample_images()["images"])
sample_images = torch.tensor(sample_images, dtype=torch.float32) / 255

In [2]:
sample_images.shape

torch.Size([2, 427, 640, 3])

In [3]:
sample_images_permuted = sample_images.permute(0, 3, 1, 2)
sample_images_permuted.shape

torch.Size([2, 3, 427, 640])

In [4]:
import torchvision
import torchvision.transforms.v2 as T
cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)
cropped_images.shape

torch.Size([2, 3, 70, 120])

In [5]:
import torch.nn as nn

torch.manual_seed(42)
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7)
fmaps = conv_layer(cropped_images)

In [6]:
fmaps.shape

torch.Size([2, 32, 64, 114])

In [7]:
conv_layer = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7, padding="same")
fmaps = conv_layer(cropped_images)
fmaps.shape

torch.Size([2, 32, 70, 120])

In [8]:
conv_layer.weight.shape

torch.Size([32, 3, 7, 7])

In [9]:
conv_layer.bias.shape

torch.Size([32])

In [10]:
max_pool = nn.MaxPool2d(kernel_size=2)
avg_pool = nn.AvgPool2d(kernel_size=2)

In [11]:
import torch.functional as F

class DepthPool(nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0):
        super().__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size
        self.padding = padding


    def forward(self, inputs):
        batch, channels, height, width = inputs.shape
        Z = inputs.view(batch, channels, height * width)  # merge spatial dims
        Z = Z.permute(0, 2, 1)  # switch spatial and channels dims
        Z = F.max_pool1d(Z, kernel_size=self.kernel_size, stride=self.stride,
                         padding=self.padding)  # compute max pool
        Z = Z.permute(0, 2, 1)  # switch back spatial and channels dims
        return Z.view(batch, -1, height, width)  # unmerge spatial dims

In [12]:
global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1)
output = global_avg_pool(cropped_images)

In [13]:
output = cropped_images.mean(dim=(2, 3), keepdim=True)

In [14]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [15]:
from functools import partial

DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same")

model = nn.Sequential(
    DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(64, 128), nn.ReLU(),
    DefaultConv2d(128, 128), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(128, 256), nn.ReLU(),
    DefaultConv2d(256, 256), nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(2304, 128), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(64, 47)  # EMNIST balanced has 47 classes
).to(device)

In [16]:
import torchmetrics
from torch.utils.data import DataLoader
from Neural_Networks_Deep_Learning.CIFAR10.utils import train

transform = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])
train_and_valid_data = torchvision.datasets.EMNIST(
    root="datasets", split="balanced", train=True, transform=transform)

test_data = torchvision.datasets.EMNIST(
    root="datasets", split="balanced", train=False, transform=transform)

train_data, valid_data = torch.utils.data.random_split(train_and_valid_data, [107800, 5000])

train_loader = DataLoader(train_data,  batch_size=32, num_workers=4, pin_memory=True, shuffle=True)
valid_loader = DataLoader(valid_data,  batch_size=32, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_data,   batch_size=32, num_workers=4, pin_memory=True)

n_epochs = 100
optimizer = torch.optim.AdamW(model.parameters())
criterion = nn.CrossEntropyLoss()
metric = torchmetrics.Accuracy(task="multiclass", num_classes=47).to(device)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer=optimizer, max_lr=1e-2, total_steps=len(train_loader)*n_epochs)
n_iter_no_improvements = 3
train(model, optimizer, criterion, train_loader, valid_loader, metric, n_epochs, n_iter_no_improvements, scheduler=scheduler)

Epoch: 1/100, Loss: 1.8130, Val Score: 0.8094
Epoch: 2/100, Loss: 0.9168, Val Score: 0.8320
Epoch: 3/100, Loss: 0.7484, Val Score: 0.8582
Epoch: 4/100, Loss: 0.6675, Val Score: 0.8566
Epoch: 5/100, Loss: 0.6236, Val Score: 0.8602
Epoch: 6/100, Loss: 0.6102, Val Score: 0.8600
Epoch: 7/100, Loss: 0.6166, Val Score: 0.8632
Epoch: 8/100, Loss: 0.6232, Val Score: 0.8564
Epoch: 9/100, Loss: 0.6508, Val Score: 0.8406
Epoch: 10/100, Loss: 0.6845, Val Score: 0.8360
Validation score has not improved for 3 epochs, stopping training


0.8632000088691711

In [19]:
from Neural_Networks_Deep_Learning.CIFAR10.utils import evaluate

evaluate(model, test_loader, metric)

0.8336170315742493